# Integración de Arquitecturas MiniRAG y TERAG para Análisis de Documentos 10-K

Este cuaderno presenta los *pipelines* base en Python para implementar las arquitecturas **MiniRAG** y **TERAG**.
Estos sistemas de *Retrieval-Augmented Generation* (RAG) están diseñados para operar con alta eficiencia en entornos locales, siendo ideales para procesar informes financieros densos como los 10-K mediante modelos de lenguaje pequeños (SLMs).

---

## 1. Arquitectura MiniRAG
**MiniRAG** está optimizado para recursos limitados. En lugar de depender de la comprensión semántica profunda de un LLM comercial, construye un **Grafo Heterogéneo Sensible a la Semántica** conectando pasajes de texto (chunks) y entidades nombradas [file:1]. En la fase de recuperación, emplea una búsqueda guiada por la topología del grafo [file:1].

In [ ]:
import networkx as nx
from sentence_transformers import SentenceTransformer

class MockSLM:
    """Mock class para simular un Modelo de Lenguaje Pequeño local (ej. Mistral o Phi-3)."""
    def extract_entities(self, text):
        # En un escenario real, esto hace un LLM prompt para NER
        return ["Net Income", "Operating Expenses", "SEC"] 
        
    def generate(self, query, context):
        return f"Generando respuesta para '{query}' basada en contexto."

class MiniRAGPipeline:
    def __init__(self, slm_model, embedding_model_name="all-MiniLM-L6-v2"):
        self.slm = slm_model
        self.embedder = SentenceTransformer(embedding_model_name)
        self.graph = nx.Graph()
        
    def index_10k_document(self, text_chunks):
        """Construcción del Grafo Heterogéneo: Conecta chunks con entidades."""
        for i, chunk in enumerate(text_chunks):
            chunk_id = f"chunk_{i}"
            self.graph.add_node(chunk_id, type="chunk", content=chunk)
            
            entities = self.slm.extract_entities(chunk)
            for entity in entities:
                self.graph.add_node(entity, type="entity")
                self.graph.add_edge(entity, chunk_id, relation="in_context")
                
    def topological_retrieval(self, query):
        """Recuperación basada en estructura del grafo (Topology-enhanced)."""
        query_entities = self.slm.extract_entities(query)
        relevant_chunks = set()
        
        for entity in query_entities:
            if entity in self.graph:
                neighbors = self.graph.neighbors(entity)
                for neighbor in neighbors:
                    if self.graph.nodes[neighbor].get('type') == 'chunk':
                        relevant_chunks.add(self.graph.nodes[neighbor]['content'])
                        
        return list(relevant_chunks)

    def pipeline_run(self, chunks, query):
        self.index_10k_document(chunks)
        context = self.topological_retrieval(query)
        return self.slm.generate(query, context)


---

## 2. Arquitectura TERAG
**TERAG (Token-Efficient Graph-Based RAG)** reduce el consumo de tokens en la indexación. Evita múltiples pasadas del LLM y extrae entidades y conceptos en una sola iteración para construir su red [file:2]. Luego, utiliza el algoritmo matemático **Personalized PageRank (PPR)** para navegar por las co-ocurrencias sin realizar llamadas adicionales al LLM en la fase de recuperación [file:2].

In [ ]:
import numpy as np

class MockExtractionLLM:
    """Mock class para simular extracción en 'few-shot' con LLM."""
    def extract_concepts(self, text):
        return ["Revenue", "Risk Factors"]

class TERAGPipeline:
    def __init__(self, extraction_llm, embedding_model="all-MiniLM-L6-v2"):
        self.llm = extraction_llm
        self.embedder = SentenceTransformer(embedding_model)
        self.graph = nx.DiGraph()
        self.concept_embeddings = {}
        
    def construct_graph(self, passages_dict):
        """Extrae conceptos en una sola pasada y crea aristas de co-ocurrencia y pasaje."""
        for p_id, text in passages_dict.items():
            self.graph.add_node(p_id, type="passage", text=text)
            
            concepts = self.llm.extract_concepts(text)
            
            for concept in concepts:
                if concept not in self.graph:
                    self.graph.add_node(concept, type="concept")
                    self.concept_embeddings[concept] = self.embedder.encode(concept)
                
                self.graph.add_edge(concept, p_id, type="has_passage")
                
            # Aristas de coocurrencia
            for i in range(len(concepts)):
                for j in range(i + 1, len(concepts)):
                    self.graph.add_edge(concepts[i], concepts[j], type="co_occurrence")
                    self.graph.add_edge(concepts[j], concepts[i], type="co_occurrence")
                    
    def ppr_retrieval(self, query, top_k=2):
        """Usa Personalized PageRank sesgado hacia los nodos de la consulta."""
        query_concepts = self.llm.extract_concepts(query)
        query_emb = self.embedder.encode(query)
        
        personalization = {node: 0.0 for node in self.graph.nodes()}
        
        for node in self.graph.nodes():
            if self.graph.nodes[node].get('type') == 'concept':
                if node in query_concepts:
                    personalization[node] = 1.0
                else:
                    sim = np.dot(query_emb, self.concept_embeddings[node])
                    personalization[node] = max(0, float(sim))
                    
        # Prevención de errores si todos son 0
        total = sum(personalization.values())
        if total > 0:
            personalization = {k: v/total for k, v in personalization.items()}
        else:
            personalization = {k: 1/len(personalization) for k in personalization}
            
        ppr_scores = nx.pagerank(self.graph, personalization=personalization, weight=None)
        
        passages_scores = {n: s for n, s in ppr_scores.items() if self.graph.nodes[n].get('type') == 'passage'}
        top_passages = sorted(passages_scores, key=passages_scores.get, reverse=True)[:top_k]
        
        return [self.graph.nodes[p_id]['text'] for p_id in top_passages]


---
**Nota:** Para ejecutar estas celdas, será necesario disponer de los paquetes `networkx`, `sentence-transformers` y `numpy` instalados en el entorno local.